In [66]:
import torch
import transformers
import peft
import datasets

In [67]:
print("Pytorch version: ",torch.__version__)
print("Cuda: ",torch.cuda.is_available())
print("Cuda version: ",torch.version.cuda)

Pytorch version:  2.14.0+cu130
Cuda:  True
Cuda version:  13.0


In [68]:
if torch.cuda.is_available():
    print("GPU: ",torch.cuda.get_device_name(0))

    vram=torch.cuda.get_device_properties(0).total_memory
    print("Vram: ",vram/1024**3,"GB")

GPU:  NVIDIA GeForce RTX 3060 Laptop GPU
Vram:  5.67022705078125 GB


In [69]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

dtype=(
    torch.bfloat16
    if torch.cuda.is_bf16_supported() else torch.float16
)

print(device)
print(dtype)


cuda
torch.bfloat16


Upload the Model 

In [70]:
from transformers import AutoTokenizer,AutoModelForCausalLM

model_name="Qwen/Qwen2.5-0.5B-Instruct"

tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(model_name,torch_dtype=torch.bfloat16)

model=model.to(device)

Checking parameters

In [71]:
total_parameters=0
trainable_parameters=0

for name,param in model.named_parameters():

    total_parameters+=param.numel()

    if param.requires_grad:

        trainable_parameters+=param.numel()

print("Total parameters: ",total_parameters)
print("Trainable parameters: ",trainable_parameters)

Total parameters:  494032768
Trainable parameters:  494032768


Checking How much get Memory

In [72]:
print("Allocated:", torch.cuda.memory_allocated()/1024**3,"GiB")
print("Reserved: ", torch.cuda.memory_reserved()/1024**3,"GiB")

Allocated: 0.9360175132751465 GiB
Reserved:  0.9921875 GiB


Model architeckture

In [73]:
#model=model.to(device)
config=model.config

print("Architekture: ",config.architectures)
print("Hidden Size: ",config.hidden_size)
print("Layers: ",config.num_hidden_layers)
print("Attention heads: ",config.num_attention_heads)
print("KV heads: ",config.num_key_value_heads)
print("Intermediate size: ",config.intermediate_size)
print("Vocabulary Size: ",config.vocab_size)
print("Context length: ",config.max_position_embeddings)

Architekture:  ['Qwen2ForCausalLM']
Hidden Size:  896
Layers:  24
Attention heads:  14
KV heads:  2
Intermediate size:  4864
Vocabulary Size:  151936
Context length:  32768


In [74]:
print(model.model.layers[0])

Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=896, out_features=896, bias=True)
    (k_proj): Linear(in_features=896, out_features=128, bias=True)
    (v_proj): Linear(in_features=896, out_features=128, bias=True)
    (o_proj): Linear(in_features=896, out_features=896, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)


In [75]:
for name, param in model.model.layers[0].named_parameters():
    print(
        name,
        "| Shape:", tuple(param.shape),
        "| Parameters:", param.numel()
    )

self_attn.q_proj.weight | Shape: (896, 896) | Parameters: 802816
self_attn.q_proj.bias | Shape: (896,) | Parameters: 896
self_attn.k_proj.weight | Shape: (128, 896) | Parameters: 114688
self_attn.k_proj.bias | Shape: (128,) | Parameters: 128
self_attn.v_proj.weight | Shape: (128, 896) | Parameters: 114688
self_attn.v_proj.bias | Shape: (128,) | Parameters: 128
self_attn.o_proj.weight | Shape: (896, 896) | Parameters: 802816
mlp.gate_proj.weight | Shape: (4864, 896) | Parameters: 4358144
mlp.up_proj.weight | Shape: (4864, 896) | Parameters: 4358144
mlp.down_proj.weight | Shape: (896, 4864) | Parameters: 4358144
input_layernorm.weight | Shape: (896,) | Parameters: 896
post_attention_layernorm.weight | Shape: (896,) | Parameters: 896


In [76]:
messages=[
    {
        "role":"user",
        "content":"O'zbekistonning poytaxti qaysi shahar?"
    }
]

prompt=tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
O'zbekistonning poytaxti qaysi shahar?<|im_end|>
<|im_start|>assistant



In [77]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False
).to(device)

print("Input IDs:", inputs["input_ids"])
print("Attention mask:",inputs["attention_mask"])
print("Shape:",inputs["input_ids"].shape)

Input IDs: tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,     46,      6,  75013,
          58819,   1229,    281,   2253,  17616,  10251,   2804,    942,     72,
            557,   1466,    277,     30, 151645,    198, 151644,  77091,    198]],
       device='cuda:0')
Attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')
Shape: torch.Size([1, 45])


In [78]:
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0].tolist()
)

print(tokens)

['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'ĠQ', 'wen', ',', 'Ġcreated', 'Ġby', 'ĠAlibaba', 'ĠCloud', '.', 'ĠYou', 'Ġare', 'Ġa', 'Ġhelpful', 'Ġassistant', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'user', 'Ċ', 'O', "'", 'zbek', 'iston', 'ning', 'Ġp', 'oy', 'tax', 'ti', 'Ġq', 'ays', 'i', 'Ġsh', 'ah', 'ar', '?', '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ']


In [100]:
model.eval()

with torch.inference_mode():

    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False
    )

input_length = inputs["input_ids"].shape[1]

new_tokens = outputs[0][input_length:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print(answer)

O'zbekistonning poytaxti qaysi shahar 10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000

In [101]:
import time
import torch

def inference(question, max_new_tokens=64):

    model.eval()

    messages = [
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(device)

    input_length = inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    torch.cuda.synchronize()

    elapsed_time = time.perf_counter() - start_time

    generated_tokens = outputs[0][input_length:]

    output_length = generated_tokens.numel()

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    tokens_per_second = output_length / elapsed_time

    peak_vram = torch.cuda.max_memory_allocated() / 1024**3

    return {
        "question": question,
        "answer": answer,
        "input_tokens": input_length,
        "output_tokens": output_length,
        "latency": round(elapsed_time, 3),
        "tokens_per_second": round(tokens_per_second, 2),
        "peak_vram_gib": round(peak_vram, 3)
    }

In [102]:
result = inference("O'zbekistonning poytaxti qaysi shahar?")

for key, value in result.items():
    print(f"{key}: {value}")

question: O'zbekistonning poytaxti qaysi shahar?
answer: O'zbekistonning poytaxti qaysi shahar 100000000000000000000000000000000000000000000000
input_tokens: 45
output_tokens: 64
latency: 0.703
tokens_per_second: 91.1
peak_vram_gib: 0.939


In [103]:
from datasets import load_dataset

dataset=load_dataset("openai/gsm8k","main")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})


In [104]:
print(dataset["train"][0]["question"])
print(dataset["train"][0]["answer"])

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [105]:
print(dataset["train"].column_names)
print(dataset["train"].features)

['question', 'answer']
{'question': Value('string'), 'answer': Value('string')}


In [106]:
train_validation=(dataset["train"]
.shuffle(seed=42).select(range(500)).train_test_split(test_size=50,seed=42))


train_dataset=train_validation["train"]
val_dataset=train_validation['test']

test_dataset=(dataset["test"].shuffle(seed=42).select(range(50)))

print("Train: ",len(train_dataset))
print("Validation: ",len(val_dataset))
print("Test: ",len(test_dataset))

Train:  450
Validation:  50
Test:  50


In [107]:
import re

def extract_answer(text):

    text = text.replace(",", "")

    match = re.search(
        r"####\s*(-?\d+(?:\.\d+)?)",
        text
    )

    if match:
        return float(match.group(1))

    return None

In [108]:
text="20+10=30. #### 30"
print(extract_answer(text))
print(extract_answer("5 + 10 = 15. #### 15"))

30.0
15.0


In [109]:
from tqdm.auto import tqdm

def evaluate(dataset):

    correct=0
    total=len(dataset)

    results=[]

    for sample in tqdm(dataset):

        question=sample["question"]
        expected=extract_answer(sample["answer"])

        prompt=(
            question
            +"\nSolve the problem step by step."
            +"\nReturn the final answer as #### number."
        )

        result=inference(
            prompt,
            max_new_tokens=256
        )

        prediction = extract_answer(result["answer"])

        is_correct = (
            prediction is not None
            and prediction == expected
        )

        if is_correct:
            correct+=1

        results.append({
            "question": question,
            "expected": expected,
            "prediction": prediction,
            "correct": is_correct,
            "response": result["answer"],
            "latency": result["latency"],
            "tokens_per_second": result["tokens_per_second"]
        })

    accuracy = correct / total * 100

    return accuracy, results

In [110]:
accuracy, results = evaluate(
    test_dataset.select(range(5))
)

print("Accuracy:", accuracy, "%")

100%|██████████| 5/5 [00:13<00:00,  2.65s/it]

Accuracy: 0.0 %


In [111]:
import pandas as pd
df=pd.DataFrame(results)

df[["expected","prediction","correct","response"]]

,expected,prediction,correct,response
0,109.0,None,False,"To solve this problem, we need to determine Al..."
1,89.0,None,False,To determine how many buttons Lorraine has at ...
2,13.0,None,False,To determine the total number of letters in bo...
3,5.0,None,False,To determine how many more laps Bethany can ru...
4,25.0,None,False,"To solve this problem, let's break down the si..."


In [112]:
for result in results:


    print("Question:", result["question"])
    print("Expected:", result["expected"])
    print("Prediction:", result["prediction"])
    print("Response:", result["response"])

    print("-" * 80)

Question: Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.
Expected: 109.0
Prediction: None
Response: To solve this problem, we need to determine Allen's current age first, given that his age-to-age ratio with Darrell is 7:11 and their combined age is 162.

Step 1: Let \( A \) be Darrell's age and \( E \) be Allen's age. According to the problem, the ratio of their ages is 7:11, so we can write:
\[ \frac{A}{E} = \frac{7}{11} \]
This implies:
\[ A = \frac{7}{11}E \]

Step 2: The sum of their ages is given as 162:
\[ A + E = 162 \]

Step 3: Substitute \( A = \frac{7}{11}E \) into the equation \( A + E = 162 \):
\[ \frac{7}{11}E + E = 162 \]

Step 4: Combine the terms on the left side:
\[ \left( \frac{7}{11} + 1 \right)E = 162 \]
\[ \left( \frac{7}{11} + \frac
--------------------------------------------------------------------------------
Question: Lorraine and Colleen are trading stickers for buttons. Each large s

In [113]:
from tqdm.auto import tqdm

def evaluate(dataset, max_new_tokens=1024):

    correct = 0
    total = len(dataset)
    results = []

    for sample in tqdm(dataset):

        question = sample["question"]
        expected = extract_answer(sample["answer"])

        prompt = (
            question
            + "\nSolve using short calculations."
            + "\nReturn the final answer as #### number."
        )

        result = inference(
            prompt,
            max_new_tokens=max_new_tokens
        )

        prediction = extract_answer(result["answer"])

        is_correct = (
            prediction is not None
            and prediction == expected
        )

        if is_correct:
            correct += 1

        results.append({
            "question": question,
            "expected": expected,
            "prediction": prediction,
            "correct": is_correct,
            "response": result["answer"],
            "latency": result["latency"],
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "tokens_per_second": result["tokens_per_second"],
            "token_limit_reached": (
                result["output_tokens"] >= max_new_tokens
            )
        })

    accuracy = correct / total * 100

    return accuracy, results

In [114]:
accuracy, results = evaluate(
    test_dataset.select(range(5)),
    max_new_tokens=1024
)

print("Accuracy:", accuracy, "%")

100%|██████████| 5/5 [00:16<00:00,  3.26s/it]

Accuracy: 0.0 %


In [115]:
import pandas as pd

df = pd.DataFrame(results)

df[[
    "expected",
    "prediction",
    "correct",
    "input_tokens",
    "output_tokens",
    "token_limit_reached",
    "latency"
]]

,expected,prediction,correct,input_tokens,output_tokens,token_limit_reached,latency
0,109.0,None,False,82,415,False,4.261
1,89.0,None,False,148,418,False,4.470
2,13.0,None,False,86,228,False,2.511
3,5.0,None,False,107,208,False,2.285
4,25.0,None,False,139,254,False,2.745


In [116]:
for result in results:

    print("Expected:", result["expected"])
    print("Response:", result["response"][-500:])
    print("-" * 80)

Expected: 109.0
Response: {11} A + A = 162 \]

Combining the terms on the left side:
\[ \left( \frac{7}{11} + 1 \right) A = 162 \]
\[ \left( \frac{7}{11} + \frac{11}{11} \right) A = 162 \]
\[ \frac{18}{11} A = 162 \]

Now, solving for \(A\):
\[ A = 162 \times \frac{11}{18} \]
\[ A = 9 \times 11 \]
\[ A = 99 \]

So, Allen's current age is 99 years.

To find out how old Allen will be 10 years from now:
\[ A_{10} = A + 10 \]
\[ A_{10} = 99 + 10 \]
\[ A_{10} = 109 \]

Therefore, Allen's age 10 years from now will be **109**.
--------------------------------------------------------------------------------
Expected: 89.0
Response: tons traded away:**
   - Total buttons traded away:
     \[
     27 + 60 = 87 \text{ buttons}
     \]

5. **Determine the number of buttons Lorraine has left after trading:**
   - Initial buttons:
     \[
     120 \text{ large buttons} + 30 \text{ small buttons} = 150 \text{ buttons}
     \]
   - Buttons traded away:
     \[
     87 \text{ buttons}
     \]
   - Rem

In [117]:
import re

def extract_answer(text):

    text = text.replace(",", "")

    patterns = [
        r"####\s*(-?\d+(?:\.\d+)?)",
        r"\\boxed\{\s*(-?\d+(?:\.\d+)?)\s*\}",
        r"(?:final answer|answer)\s*(?:is|:|=)?\s*\**\$?\s*(-?\d+(?:\.\d+)?)"
    ]

    for pattern in patterns:

        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if matches:
            return float(matches[-1])

    return None

In [118]:
print(extract_answer("#### 109"))
print(extract_answer(r"\boxed{109}"))
print(extract_answer("The final answer is 109."))
print(extract_answer("Answer: 109"))

109.0
109.0
109.0
109.0


In [119]:
for result in results:

    result["prediction"] = extract_answer(
        result["response"]
    )

    result["correct"] = (
        result["prediction"] is not None
        and result["prediction"] == result["expected"]
    )

df = pd.DataFrame(results)

accuracy = df["correct"].mean() * 100

print("Accuracy:", accuracy, "%")

df[["expected", "prediction", "correct"]]

Accuracy: 40.0 %


,expected,prediction,correct
0,109.0,NaN,False
1,89.0,63.0,False
2,13.0,13.0,True
3,5.0,5.0,True
4,25.0,40.0,False
